06 Task-Aware Evaluation

Goal: evaluate pin quality beyond median offset. This notebook introduces movable-place evaluation, arrival-cost scoring, and optional comparison for future repositioning methods.

In [17]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.features import place_complexity, pin_ambiguity, should_move_rule
from src.metrics import (
    haversine_meters,
    task_aware_report,
    segmented_task_report,
    arrival_cost_score,
)

PROCESSED = PROJECT_ROOT / "data" / "processed"

PROJECT_ROOT


PosixPath('/Users/shivanibelambe/Pin-To-Place')

In [18]:
combined_path = PROCESSED / "ground_truth_combined.csv"

if combined_path.exists():
    df = pd.read_csv(combined_path)
else:
    files = sorted(PROCESSED.glob("ground_truth_*.csv"))
    df = pd.concat([pd.read_csv(path) for path in files], ignore_index=True)

df["place_complexity"] = df.get("place_complexity", df.apply(place_complexity, axis=1))
df["pin_ambiguity"] = df.get("pin_ambiguity", df.apply(pin_ambiguity, axis=1))
df["should_move"] = df.get("should_move", df.apply(should_move_rule, axis=1))

df.shape


(6850, 35)

In [19]:
baseline_report = task_aware_report(df)

baseline_report


{'count': 6836,
 'mean_m': np.float64(2.69),
 'median_m': np.float64(0.0),
 'p90_m': np.float64(20.02),
 'p95_m': np.float64(23.34),
 'max_m': np.float64(74.77),
 'pct_exact_no_move': np.float64(88.5),
 'pct_over_10m': np.float64(11.5),
 'pct_over_25m': np.float64(3.1),
 'pct_over_50m': np.float64(0.0)}

In [20]:
movable_df = df[df["should_move"]].copy()
protected_df = df[~df["should_move"]].copy()

{
    "all_rows": len(df),
    "movable_rows": len(movable_df),
    "protected_rows": len(protected_df),
    "movable_pct": round(len(movable_df) / len(df) * 100, 1),
}


{'all_rows': 6850,
 'movable_rows': 784,
 'protected_rows': 6066,
 'movable_pct': 11.4}

In [21]:
{
    "all_places": task_aware_report(df),
    "movable_places": task_aware_report(movable_df) if len(movable_df) else {},
    "protected_places": task_aware_report(protected_df) if len(protected_df) else {},
}


{'all_places': {'count': 6836,
  'mean_m': np.float64(2.69),
  'median_m': np.float64(0.0),
  'p90_m': np.float64(20.02),
  'p95_m': np.float64(23.34),
  'max_m': np.float64(74.77),
  'pct_exact_no_move': np.float64(88.5),
  'pct_over_10m': np.float64(11.5),
  'pct_over_25m': np.float64(3.1),
  'pct_over_50m': np.float64(0.0)},
 'movable_places': {'count': 784,
  'mean_m': np.float64(23.4),
  'median_m': np.float64(22.95),
  'p90_m': np.float64(27.64),
  'p95_m': np.float64(30.3),
  'max_m': np.float64(74.77),
  'pct_exact_no_move': np.float64(0.0),
  'pct_over_10m': np.float64(100.0),
  'pct_over_25m': np.float64(27.3),
  'pct_over_50m': np.float64(0.3)},
 'protected_places': {'count': 6052,
  'mean_m': np.float64(0.0),
  'median_m': np.float64(0.0),
  'p90_m': np.float64(0.0),
  'p95_m': np.float64(0.0),
  'max_m': np.float64(9.8),
  'pct_exact_no_move': np.float64(100.0),
  'pct_over_10m': np.float64(0.0),
  'pct_over_25m': np.float64(0.0),
  'pct_over_50m': np.float64(0.0)}}

In [22]:
segmented_task_report(df, "tier_label")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
2,418,7.34,0.0,23.13,24.44,32.98,67.5,32.5,4.8,0.0,open_space
3,4600,3.29,0.0,21.65,24.36,74.77,86.0,14.0,4.2,0.0,standard_commercial
0,326,0.43,0.0,0.00,0.00,26.97,98.2,1.8,0.6,0.0,multi_tenant
1,1492,0.00,0.0,0.00,0.00,0.00,100.0,0.0,0.0,0.0,no_building


In [23]:
segmented_task_report(df, "place_complexity")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
1,244,3.66,0.0,22.11,25.03,31.87,84.4,15.6,5.7,0.0,multi_tenant
0,604,4.37,0.0,22.46,24.15,33.65,81.1,18.9,4.0,0.0,complex
2,5988,2.48,0.0,18.57,23.19,74.77,89.4,10.6,2.9,0.0,simple


In [24]:
segmented_task_report(df, "pin_ambiguity")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
1,4116,3.46,0.0,21.78,23.87,74.77,85.2,14.8,4.1,0.0,low
2,662,2.22,0.0,0.00,22.93,35.84,90.6,9.4,3.0,0.0,medium
0,2058,1.28,0.0,0.00,18.63,33.65,94.5,5.5,1.2,0.0,high


In [25]:
def infer_arrival_friction(row) -> dict:
    """
    V1 heuristic arrival-friction labels.

    These are intentionally conservative placeholders until you have
    sidewalk, curb-cut, road-network, parking-lot, or imagery-derived features.
    """
    tier = row.get("tier_label")
    complexity = row.get("place_complexity")
    category = str(row.get("category_primary", "")).lower()

    parking_lot_crossing = tier in {"standard_commercial", "multi_tenant"} and complexity in {
        "complex",
        "multi_tenant",
    }

    sidewalk_visible = None
    barrier_detected = False

    if tier == "open_space":
        sidewalk_visible = False

    if category in {"campground", "rv_park", "resort"}:
        parking_lot_crossing = True

    return {
        "sidewalk_visible": sidewalk_visible,
        "parking_lot_crossing": parking_lot_crossing,
        "barrier_detected": barrier_detected,
    }


friction = df.apply(infer_arrival_friction, axis=1, result_type="expand")
df = pd.concat([df, friction], axis=1)

df[["sidewalk_visible", "parking_lot_crossing", "barrier_detected"]].head()

,sidewalk_visible,parking_lot_crossing,barrier_detected
0,None,False,False
1,None,False,False
2,None,False,False
3,None,False,False
4,None,False,False


In [26]:
df["arrival_cost_m"] = df.apply(
    lambda row: arrival_cost_score(
        distance_m=row["offset_haversine_m"],
        sidewalk_visible=row["sidewalk_visible"],
        parking_lot_crossing=row["parking_lot_crossing"],
        barrier_detected=row["barrier_detected"],
    ),
    axis=1,
)

task_aware_report(df, offset_col="arrival_cost_m")

{'count': 6836,
 'mean_m': np.float64(4.64),
 'median_m': np.float64(0.0),
 'p90_m': np.float64(22.2),
 'p95_m': np.float64(25.34),
 'max_m': np.float64(74.77),
 'pct_exact_no_move': np.float64(77.9),
 'pct_over_10m': np.float64(15.6),
 'pct_over_25m': np.float64(5.3),
 'pct_over_50m': np.float64(0.2)}

In [27]:
segmented_task_report(df, "tier_label", offset_col="arrival_cost_m")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
2,418,27.36,25.0,47.15,48.34,57.98,0.0,100.0,32.5,2.4,open_space
3,4600,4.25,0.0,21.78,24.96,74.77,77.8,14.0,4.9,0.0,standard_commercial
0,326,2.33,0.0,10.00,10.00,26.97,79.1,1.8,0.6,0.0,multi_tenant
1,1492,0.00,0.0,0.00,0.00,0.00,100.0,0.0,0.0,0.0,no_building


In [28]:
segmented_task_report(df, "place_complexity", offset_col="arrival_cost_m")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
0,604,19.02,10.0,45.22,47.91,57.98,6.3,39.7,18.9,1.7,complex
1,244,10.01,10.0,32.11,35.03,41.87,36.9,16.4,14.8,0.0,multi_tenant
2,5988,2.97,0.0,18.63,23.56,74.77,86.8,13.1,3.5,0.0,simple


In [29]:
def compute_method_offset(df, lat_col, lon_col, output_col):
    result = df.copy()

    valid = result[[lat_col, lon_col, "gt_lat", "gt_lon"]].notna().all(axis=1)

    result[output_col] = np.nan
    result.loc[valid, output_col] = result.loc[valid].apply(
        lambda row: haversine_meters(
            row[lat_col],
            row[lon_col],
            row["gt_lat"],
            row["gt_lon"],
        ),
        axis=1,
    )

    return result


candidate_methods = {
    "ensemble": ("ensemble_lat", "ensemble_lon"),
    "ranker": ("ranker_lat", "ranker_lon"),
    "llm": ("llm_lat", "llm_lon"),
}

available_methods = {
    name: cols
    for name, cols in candidate_methods.items()
    if cols[0] in df.columns and cols[1] in df.columns
}

available_methods

{}

In [30]:
method_reports = []

for method_name, (lat_col, lon_col) in available_methods.items():
    offset_col = f"{method_name}_offset_m"
    df = compute_method_offset(df, lat_col, lon_col, offset_col)

    baseline = df["offset_haversine_m"]
    method = df[offset_col]

    valid = method.notna()
    regression_rate = round((method[valid] > baseline[valid]).mean() * 100, 2)

    report = task_aware_report(df[valid], offset_col=offset_col)
    report["method"] = method_name
    report["regression_rate_pct"] = regression_rate
    method_reports.append(report)

if method_reports:
    pd.DataFrame(method_reports).sort_values("p95_m")
else:
    "No repositioning method columns found yet. This is expected until ensemble/ranker/LLM outputs are generated."

In [31]:
evaluation_cols = [
    "id",
    "name",
    "category_primary",
    "region",
    "tier_label",
    "place_complexity",
    "pin_ambiguity",
    "should_move",
    "offset_haversine_m",
    "arrival_cost_m",
    "gt_confidence",
    "sidewalk_visible",
    "parking_lot_crossing",
    "barrier_detected",
]

task_eval = df[evaluation_cols].copy()
task_eval.to_csv(PROCESSED / "task_aware_evaluation.csv", index=False)

task_eval.sort_values("arrival_cost_m", ascending=False).head(50)

,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,should_move,offset_haversine_m,arrival_cost_m,gt_confidence,sidewalk_visible,parking_lot_crossing,barrier_detected
6501,08f489c0a35ab63203e9b0cd0f7c2d30,Subway,sandwich_shop,TX,standard_commercial,simple,low,True,74.772429,74.77,0.9,None,False,False
3076,08f489c0a35ab63203e9b0cd0f7c2d30,Subway,sandwich_shop,TX,standard_commercial,simple,low,True,74.772429,74.77,0.9,None,False,False
3456,08f489d59ab606c803517ccee128ff0f,Chateau Burg RV Resort,rv_park,TX,open_space,complex,high,True,32.981592,57.98,0.9,False,True,False
31,08f489d59ab606c803517ccee128ff0f,Chateau Burg RV Resort,rv_park,TX,open_space,complex,high,True,32.981592,57.98,0.9,False,True,False
4765,08f4468143b34d0d03cca189ab50c32b,Arrowhead RV Park,rv_park,TX,open_space,complex,high,True,32.812913,57.81,0.9,False,True,False
1340,08f4468143b34d0d03cca189ab50c32b,Arrowhead RV Park,rv_park,TX,open_space,complex,high,True,32.812913,57.81,0.9,False,True,False
5794,08f26c82d56f5b9303c8d5ac1586d26a,Plantation Place Dallas RV Park,rv_park,TX,open_space,complex,high,True,32.077439,57.08,0.9,False,True,False
2369,08f26c82d56f5b9303c8d5ac1586d26a,Plantation Place Dallas RV Park,rv_park,TX,open_space,complex,high,True,32.077439,57.08,0.9,False,True,False
3331,08f266ddac22bad803f4cb082276e7ef,Swiss Haven RV Resort,rv_park,IN,open_space,complex,high,True,29.788903,54.79,0.9,False,True,False
6756,08f266ddac22bad803f4cb082276e7ef,Swiss Haven RV Resort,rv_park,IN,open_space,complex,high,True,29.788903,54.79,0.9,False,True,False


In [32]:
summary_lines = []

summary_lines.append("Task-Aware Evaluation Summary")
summary_lines.append("")
summary_lines.append("Baseline geometric offset")
summary_lines.append(str(task_aware_report(df, offset_col="offset_haversine_m")))
summary_lines.append("")
summary_lines.append("Arrival-cost score")
summary_lines.append(str(task_aware_report(df, offset_col="arrival_cost_m")))
summary_lines.append("")
summary_lines.append("Arrival cost by tier")
summary_lines.append(segmented_task_report(df, "tier_label", offset_col="arrival_cost_m").to_string(index=False))
summary_lines.append("")
summary_lines.append("Arrival cost by complexity")
summary_lines.append(segmented_task_report(df, "place_complexity", offset_col="arrival_cost_m").to_string(index=False))

summary_path = PROCESSED / "task_aware_evaluation_summary.txt"
summary_path.write_text("\n".join(summary_lines))

summary_path

PosixPath('/Users/shivanibelambe/Pin-To-Place/data/processed/task_aware_evaluation_summary.txt')

First, place_complexity separates simple places from multi-tenant and complex places. This makes the analysis fairer because a campground, resort, or shopping center should not be judged the same way as a standalone pizza shop.

Second, pin_ambiguity marks whether a place has one obvious pin or several plausible targets. This gives the project a more research-like angle: sometimes the problem is not “bad coordinate,” it is “the coordinate is inherently ambiguous.”

Third, should_move turns repositioning into a conservative decision. Since your current median offset is already 0.0m, the model should first decide whether a pin deserves movement at all.

Fourth, arrival_cost_m starts moving the project beyond raw distance. It is a v1 score for practical arrival friction: not just “how far is the pin from the label,” but “does this pin make arrival harder?”